In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import os, gc, soundfile as sf, librosa
from joblib import Parallel, delayed

# =============================================================================
# CONFIGS
# =============================================================================
IMG_SIZE              = (128, 313)
SAMPLE_RATE           = 32000
DURATION              = 7.0
STEP                  = 5.0           # restored from original
BATCH_SIZE            = 32
THRESHOLD             = 0.01
VETO_CUTOFF           = 0.60
UNCERTAINTY_THRESHOLD = 0.30
PARTIAL_PATH          = 'submission_partial.csv'

# =============================================================================
# CLEAN SLATE
# =============================================================================
if os.path.exists(PARTIAL_PATH):
    os.remove(PARTIAL_PATH)
    print("Cleared previous partial CSV.")

# =============================================================================
# CPU THREADING
# =============================================================================
tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)

# =============================================================================
# PATHS
# =============================================================================
CONV_MODEL_PATH   = "/kaggle/input/notebooks/enoughrook2/convnext-birdclf-train/convnext_bird_model.keras"
CONV_CLASSES_PATH = "/kaggle/input/notebooks/enoughrook2/birdclef-audio-decode/bird_names.npy"
PERCH_PATH        = "/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8"
SAMPLE_SUB_PATH   = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"
TEST_DIR          = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"

# =============================================================================
# LOAD MODELS
# =============================================================================
print("Loading ConvNeXt...")
model = tf.keras.models.load_model(CONV_MODEL_PATH, compile=False)
model_classes = np.load(CONV_CLASSES_PATH).tolist()
print(f"ConvNeXt loaded — {len(model_classes)} classes.")

print("Loading Perch...")
perch_model = hub.KerasLayer(
    PERCH_PATH,
    signature="serving_default",
    signature_outputs_as_dict=True
)
perch_labels = pd.read_csv(f"{PERCH_PATH}/assets/label.csv")
print(f"Perch loaded — {len(perch_labels)} labels.")

# =============================================================================
# tf.functions
# =============================================================================
@tf.function(reduce_retracing=True)
def run_conv(x):
    return model(x, training=False)

@tf.function(reduce_retracing=True)
def run_perch(x):
    return perch_model(x)

# =============================================================================
# MAPPING LOGIC
# =============================================================================
perch_to_comp_idx = np.full(len(perch_labels), -1)
for i, row in perch_labels.iterrows():
    if row['ebird2021'] in model_classes:
        perch_to_comp_idx[i] = model_classes.index(row['ebird2021'])

non_bird_indices = perch_labels[perch_labels['ebird2021'].isna()].index.tolist()

sample_sub      = pd.read_csv(SAMPLE_SUB_PATH)
sub_col_indices = [i for i, bird in enumerate(model_classes) if bird in sample_sub.columns]
sub_col_names   = [model_classes[i] for i in sub_col_indices]

print(f"Non-bird veto indices: {len(non_bird_indices)}")

# =============================================================================
# HELPERS
# =============================================================================
def ensure_length(audio, target_samples):
    if len(audio) < target_samples:
        return np.pad(audio, (0, target_samples - len(audio)), mode='constant')
    return audio[:target_samples]

def compute_single_mel(chunk):
    """Original mel preprocessing — no pre-emphasis, matches training."""
    if np.abs(chunk).max() < 1e-6:
        return np.zeros((128, 313))
    spec    = librosa.feature.melspectrogram(
        y=chunk, sr=SAMPLE_RATE, n_mels=128, fmin=40, fmax=15000
    )
    spec_db = librosa.power_to_db(spec, ref=np.max)
    mn, mx  = spec_db.min(), spec_db.max()
    if mx - mn < 1e-6:
        return np.zeros((128, 313))
    return (spec_db - mn) / (mx - mn + 1e-6)

def batch_preprocess_conv(chunks_np):
    """Parallel mel computation. Returns (B,128,313,3) tensor."""
    specs = Parallel(n_jobs=-1, prefer="threads")(
        delayed(compute_single_mel)(chunks_np[i])
        for i in range(len(chunks_np))
    )
    specs = np.stack(specs)[..., np.newaxis]
    t = tf.image.resize(specs, IMG_SIZE)
    return tf.tile(t, [1, 1, 1, 3])

def apply_perch_veto(b_audio, c_probs, final_b):
    """
    Veto only — no score blending.
    Perch only suppresses non-bird chunks on uncertain predictions.
    """
    uncertain_indices = np.where(c_probs.max(axis=1) < UNCERTAINTY_THRESHOLD)[0]

    if len(uncertain_indices) == 0:
        return final_b

    uncertain_audio = b_audio[uncertain_indices]
    perch_in = np.stack([
        ensure_length(a, 160000) for a in uncertain_audio
    ]).astype(np.float32)

    p_out   = run_perch(perch_in)
    p_probs = tf.nn.sigmoid(p_out['label']).numpy()

    # Veto only
    v_scores  = p_probs[:, non_bird_indices].max(axis=1)
    veto_hits = uncertain_indices[v_scores > VETO_CUTOFF]
    if len(veto_hits) > 0:
        final_b[veto_hits] *= 0.1

    return final_b

def flush_results(results):
    if not results:
        return []
    write_header = not os.path.exists(PARTIAL_PATH)
    pd.DataFrame(results).to_csv(
        PARTIAL_PATH, mode='a', header=write_header, index=False
    )
    return []

# =============================================================================
# INFERENCE
# =============================================================================
test_files = sorted([
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR) if f.endswith('.ogg')
])
print(f"Found {len(test_files)} test files.")

all_results = []
f_step = int(STEP * SAMPLE_RATE)
f_win  = int(DURATION * SAMPLE_RATE)

for file_path in test_files:
    file_id = os.path.splitext(os.path.basename(file_path))[0]
    try:
        try:
            audio, _ = sf.read(file_path)
        except Exception:
            audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)

        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        total_frames  = len(audio)
        file_chunks   = []
        file_row_ids  = []

        # Restored: reflection padding + 1s lookback, matches training
        for start_frame in range(0, total_frames, f_step):
            row_id    = f"{file_id}_{int((start_frame / SAMPLE_RATE) + 5)}"
            win_start = int(start_frame - (1.0 * SAMPLE_RATE))
            win_end   = win_start + f_win

            if win_start < 0:
                chunk = np.pad(audio[0:max(0, win_end)], (abs(win_start), 0), mode='reflect')
            elif win_end > total_frames:
                chunk = np.pad(audio[win_start:], (0, win_end - total_frames), mode='reflect')
            else:
                chunk = audio[win_start:win_end]

            if len(chunk) == f_win:
                file_chunks.append(chunk)
                file_row_ids.append(row_id)

        for i in range(0, len(file_chunks), BATCH_SIZE):
            try:
                b_audio = np.array(file_chunks[i:i + BATCH_SIZE], dtype=np.float32)
                b_ids   = file_row_ids[i:i + BATCH_SIZE]

                conv_input = batch_preprocess_conv(b_audio)
                c_probs    = run_conv(conv_input).numpy()

                if np.isnan(c_probs).any():
                    c_probs = np.nan_to_num(c_probs, nan=0.0)

                final_b = c_probs.copy()
                final_b = apply_perch_veto(b_audio, c_probs, final_b)
                final_b[final_b < THRESHOLD] = 0.0

                for j, row_probs in enumerate(final_b):
                    res = {'row_id': b_ids[j]}
                    res.update(dict(zip(sub_col_names, row_probs[sub_col_indices])))
                    all_results.append(res)

            except Exception as e:
                print(f"Skipping batch {i} in {file_id}: {e}")
                continue

        all_results = flush_results(all_results)
        del audio, file_chunks
        gc.collect()

    except Exception as e:
        print(f"Skipping file {file_id}: {e}")
        continue

flush_results(all_results)

# =============================================================================
# SUBMISSION
# =============================================================================
if os.path.exists(PARTIAL_PATH):
    partial = pd.read_csv(PARTIAL_PATH)
    partial = partial.drop_duplicates(subset='row_id', keep='last')
    print(f"Partial CSV: {len(partial)} unique rows.")

    for col in sample_sub.columns:
        if col not in partial.columns:
            partial[col] = 0.0

    bird_cols = [c for c in sample_sub.columns if c != 'row_id']
    partial[bird_cols] = partial[bird_cols].clip(0.0, 1.0)

    (
        partial
        .merge(sample_sub[['row_id']], on='row_id', how='right')
        .fillna(0.0)
        [sample_sub.columns]
        .to_csv('submission.csv', index=False)
    )
    print("submission.csv saved.")
else:
    sample_sub.to_csv('submission.csv', index=False)
    print("No results — saved sample submission.")

2026-04-21 02:10:56.921870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776737457.167163      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776737457.236786      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776737457.844638      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776737457.844690      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776737457.844695      16 computation_placer.cc:177] computation placer alr

Loading ConvNeXt...


2026-04-21 02:11:30.155645: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ConvNeXt loaded — 206 classes.
Loading Perch...
Perch loaded — 10932 labels.
Non-bird veto indices: 0
Found 0 test files.
No results — saved sample submission.
